In [1]:
# Keep-alive: moves mouse every 30 seconds via JavaScript
from IPython.display import display, Javascript

display(Javascript("""
function clickRun() {
    console.log("keep-alive ping");
}
setInterval(clickRun, 30000);
"""))

<IPython.core.display.Javascript object>

In [ ]:
import subprocess, sys, os, json, shutil, glob, time
from pathlib import Path

# ================= CONFIGURATION =================
DATA_BASE = "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA" # Adjust path if needed
WORKING_DIR = "/kaggle/working/project4"
PROJECT_DIR = os.path.join(WORKING_DIR, "aasist")
BACKUP_DIR = "/kaggle/working/models_backup"
LOG_PATH = os.path.join(WORKING_DIR, "training.log")
TARGET_EPOCHS = 30
# =================================================

# 1. Environment Setup
print("── Setting up environment...")
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "scikit-learn", "torchcontrib"], check=True)

# 2. Check Dataset
print("── Verifying dataset...")
required = ["ASVspoof2019_LA_train/flac", "ASVspoof2019_LA_dev/flac", "ASVspoof2019_LA_cm_protocols"]
for r in required:
    path = os.path.join(DATA_BASE, r)
    if not os.path.exists(path):
        print(f"  ⚠ WARNING: {path} not found. Searching for dataset...")
        # Fallback search if path is different
        search = glob.glob(f"/kaggle/input/**/{r}", recursive=True)
        if search: DATA_BASE = search[0].replace("/" + r, ""); break
        else: raise FileNotFoundError(f"Dataset root containing {r} not found.")

# 3. Clone and Prepare AASIST
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

if not os.path.exists(PROJECT_DIR):
    print("── Cloning AASIST...")
    subprocess.run(["git", "clone", "https://github.com/clovaai/aasist.git", PROJECT_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements.txt"], check=True)

# 4. Critical Patches
print("── Applying patches...")

# Patch 4.1: DataLoader num_workers (Avoid GPU multiprocessing issues on Kaggle)
for file in glob.glob(f"{PROJECT_DIR}/**/*.py", recursive=True):
    with open(file, "r") as f: content = f.read()
    if "DataLoader(" in content:
        content = content.replace("num_workers=4", "num_workers=0").replace("num_workers=8", "num_workers=0")
        with open(file, "w") as f: f.write(content)

# Patch 4.2: Fix np.float -> np.float64 (NumPy 1.24+ compatibility)
eval_path = os.path.join(PROJECT_DIR, "evaluation.py")
with open(eval_path, "r") as f:
    content = f.read().replace(".astype(np.float)", ".astype(np.float64)")
with open(eval_path, "w") as f: f.write(content)

# Patch 4.3: Inject Resume/Checkpoint Logic into main.py
main_path = os.path.join(PROJECT_DIR, "main.py")
with open(main_path, "r") as f: lines = f.readlines()

new_lines = []
skip_until = -1
for i, line in enumerate(lines):
    if i < skip_until: continue
    
    # Inicject Resume Logic before the training loop
    if 'for epoch in range(config["num_epochs"]):' in line:
        indent = line[:line.find('for')]
        new_lines.append(f"{indent}checkpoint_path = model_save_path / 'last_checkpoint.pth'\n")
        new_lines.append(f"{indent}start_epoch = 0\n")
        new_lines.append(f"{indent}if checkpoint_path.exists():\n")
        new_lines.append(f"{indent}    print('Found checkpoint! Resuming...')\n")
        new_lines.append(f"{indent}    ckpt = torch.load(checkpoint_path, weights_only=False)\n")
        new_lines.append(f"{indent}    model.load_state_dict(ckpt['model'])\n")
        new_lines.append(f"{indent}    optimizer.load_state_dict(ckpt['optimizer'])\n")
        new_lines.append(f"{indent}    scheduler.load_state_dict(ckpt['scheduler'])\n")
        new_lines.append(f"{indent}    start_epoch = ckpt['epoch'] + 1\n")
        new_lines.append(f"{indent}for epoch in range(start_epoch, config['num_epochs']):\n")
        continue
    
    # Inject State Saving at the end of each epoch
    if 'writer.add_scalar("best_dev_tdcf", best_dev_tdcf, epoch)' in line:
        new_lines.append(line)
        indent = line[:line.find('writer')]
        new_lines.append(f"{indent}torch.save({{\n")
        new_lines.append(f"{indent}    'epoch': epoch, 'model': model.state_dict(),\n")
        new_lines.append(f"{indent}    'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict()\n")
        new_lines.append(f"{indent}}}, checkpoint_path)\n")
        new_lines.append(f"{indent}print(f'Epoch {{epoch}} saved to {{checkpoint_path}}')\n")
        continue
        
    new_lines.append(line)

with open(main_path, "w") as f: f.writelines(new_lines)

# 5. Patch Config
config_path = os.path.join(PROJECT_DIR, "config/AASIST.conf")
with open(config_path) as f: config = json.load(f)
config.update({
    "database_path": DATA_BASE + "/",
    "batch_size": 16,
    "num_epochs": TARGET_EPOCHS,
    "num_workers": 0
})
with open(config_path, "w") as f: json.dump(config, f, indent=2)

# 6. Checkpoint Discovery (Restore from Previous Run)
# If you attached a previous run's dataset, look for checkpoints there
expected_ckpt_rel = f"LA_AASIST_ep{TARGET_EPOCHS}_bs16/weights/last_checkpoint.pth"
local_ckpt_path = os.path.join(PROJECT_DIR, "exp_result", expected_ckpt_rel)

if not os.path.exists(local_ckpt_path):
    print("── Searching for existing checkpoints to restore...")
    found_ckpts = glob.glob(f"/kaggle/input/**/last_checkpoint.pth", recursive=True)
    if found_ckpts:
        print(f"  ✓ found checkpoint at {found_ckpts[0]}")
        os.makedirs(os.path.dirname(local_ckpt_path), exist_ok=True)
        shutil.copy(found_ckpts[0], local_ckpt_path)

# Quick patch — fix torch.load in the already-cloned main.py
main_path = "/kaggle/working/project4/aasist/main.py"

with open(main_path, "r") as f:
    content = f.read()

# Fix the checkpoint load line
content = content.replace(
    "ckpt = torch.load(checkpoint_path)",
    "ckpt = torch.load(checkpoint_path, weights_only=False)"
)

with open(main_path, "w") as f:
    f.write(content)

print("✓ Fixed torch.load weights_only in main.py")


# 7. Start Training
print(f"\n============================================")
print(f"  TRAINING STARTED (Target: {TARGET_EPOCHS} Epochs)")
print(f"  Log: {LOG_PATH}")
print(f"============================================\n")

os.chdir(PROJECT_DIR)
with open(LOG_PATH, "a") as log:
    process = subprocess.Popen(
        [sys.executable, "-u", "main.py", "--config", "./config/AASIST.conf"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    
    last_output_time = time.time()
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()
        last_output_time = time.time()
        
        # Periodic Heartbeat (keep Kaggle alive if it gets too quiet)
        if time.time() - last_output_time > 60:
            print(".", end="")
            last_output_time = time.time()

process.wait()

# 8. Post-Run Backup
if os.path.exists("exp_result"):
    print("\n── Backing up all results to /kaggle/working/models_backup...")
    # Clean old backup if it exists to avoid nested copytree
    if os.path.exists(BACKUP_DIR): shutil.rmtree(BACKUP_DIR)
    shutil.copytree("exp_result", BACKUP_DIR)
    print(f"✓ Backup complete. Files: {os.listdir(BACKUP_DIR)}")

print("\n============================================")
print("  PROCESS COMPLETE")
print("============================================")


── Setting up environment...
── Verifying dataset...
── Applying patches...
✓ Fixed torch.load weights_only in main.py

  TRAINING STARTED (Target: 30 Epochs)
  Log: /kaggle/working/project4/training.log

2026-04-08 17:02:08.305598: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775667728.329280     299 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775667728.337036     299 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775667728.357151     299 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775667728.357211     299 computation_placer.